# Resource Estimation for Simulating a 2D Ising Model Hamiltonian

In this notebook we demonstrate how to estimate the resources for quantum
dynamics, specifically the simulation of an Ising model Hamiltonian on an
$N \times N$ 2D lattice using a *fourth-order Trotter–Suzuki product formula*
assuming a 2D qubit architecture with nearest-neighbor connectivity.

We use **qdk-chemistry** for domain-specific functionality (model
Hamiltonians, Trotter decomposition, circuit construction) and
**qsharp.qre** exclusively for quantum resource estimation.

## Background: 2D Ising Model

The Ising model is a mathematical model of ferromagnetism in a lattice
(in our case a 2D square lattice) with two kinds of terms in the
Hamiltonian: (i) an interaction term between adjacent sites and (ii) an
external magnetic field acting at each site. We consider a simplified
version where the interaction terms have the same strength and the
external field strength is the same at each site.

Formally, the Ising model Hamiltonian on an $N \times N$ lattice is:

$$
H = \underbrace{-J \sum_{\langle i, j \rangle} Z_i Z_j}_{B} - \underbrace{h \sum_j X_j}_{A}
$$

where $J$ is the interaction strength and $h$ is the external field strength.

## Creating the Ising Model

We use `qdk_chemistry` to define the lattice geometry and build the
Hamiltonian.  `LatticeGraph.square` creates a 2D square lattice and
`create_ising_hamiltonian` produces the corresponding `QubitHamiltonian`
with ZZ interaction and transverse X field terms.

In [ ]:
from qdk_chemistry.data import LatticeGraph
from qdk_chemistry.utils.model_hamiltonians import create_ising_hamiltonian

lattice = LatticeGraph.square(10, 10, t=1.0)
hamiltonian = create_ising_hamiltonian(lattice, j=1.0, h=0.5)

print(f"Ising model on {lattice.num_sites}-site lattice "
      f"with {lattice.num_edges} edges")
print(f"QubitHamiltonian: {len(hamiltonian.pauli_strings)} Pauli terms, "
      f"{hamiltonian.num_qubits} qubits")

## Trotter Expansion

The time evolution $e^{-iHt}$ for the Hamiltonian is simulated with the
fourth-order product formula so that any errors in simulation are
sufficiently small. Essentially, this is done by simulating the evolution
for small slices of time $\Delta$ and repeating this for `nSteps`
$= t/\Delta$ to obtain the full time evolution. The Trotter–Suzuki
formula for higher orders can be recursively defined using a *fractal
decomposition*. In particular, the fourth-order formula $U_4(\Delta)$
can be constructed using the second-order one $U_2(\Delta)$ as follows.

$$
\begin{aligned}
U_2(\Delta) & = e^{-iA\Delta/2} e^{-iB\Delta} e^{-iA\Delta/2}; \\
U_4(\Delta) & = U_2(p\Delta)U_2(p\Delta)U_2((1 - 4p)\Delta)U_2(p\Delta)U_2(p\Delta); \\
p & = (4 - 4^{1/3})^{-1}.
\end{aligned}
$$

The `Trotter` builder in qdk-chemistry implements arbitrary-order
Suzuki decompositions and automatically determines the number of
Trotter steps for a given target accuracy (or accepts an explicit
step count).

In [ ]:
import math
from qdk_chemistry.algorithms.time_evolution.builder.trotter import Trotter

t = 1.0   # total simulation time
dt = 0.5  # time per Trotter step

trotter = Trotter(order=4, num_divisions=math.ceil(t / dt), optimize_term_ordering=True)
evolution = trotter.run(hamiltonian, time=t)

container = evolution.get_container()
print(f"4th-order Trotter: {len(container.step_terms)} Pauli "
      f"exponentials per step, {container.step_reps} repetitions")
print(f"Qubits: {evolution.get_num_qubits()}")

## Building the Circuit

The `TimeEvolutionUnitary` is converted to a `Circuit` object which holds
a Cirq representation of the Trotter decomposition using
`PauliStringPhasor` operations. This circuit can then be passed to QRE's
`CirqApplication` for resource estimation.

In [ ]:
circuit = evolution.to_circuit()
cirq_circuit = circuit.get_cirq_circuit()

print(f"Cirq circuit with {len(cirq_circuit.all_operations())} operations")

## Application Wrapper

Presently the circuit is Cirq code and can be used for any purpose (e.g.
simulation). To obtain resource estimates we wrap this in a QRE
`Application` instance. The `CirqApplication` subclass does this
automatically from any `cirq.Circuit` instance.

In [ ]:
from qsharp.qre.application import CirqApplication

app = CirqApplication(cirq_circuit)
app_query = app.q()

app

## Majorana Architecture

A QRE `Architecture` is a container for an Instruction Set Architecture
(ISA): a list of instructions the hardware supports. The Majorana
architecture supports single-qubit and two-qubit measurements in the X-
and Z-bases, as well as a timing-based T-gate. The `Majorana` subclass
provides the ISA for a user-specified measurement error rate.

In [ ]:
from qsharp.qre.models import Majorana

arch = Majorana(error_rate=1e-5)
arch

## Creating Application and ISA Queries

The core of the system is modeled through trace queries (from the top) and
ISA queries (from the bottom).

The trace query applies layouts (`ISATransform` instances) to convert the
operations from the application into logical operations supported by error
correction codes and magic state factories. Here we expand fine rotation
gates into T-gates using circuit synthesis. The `PSSPC` layout takes as an
argument the options for the number of T-gates used per rotation. QRE will
enumerate over this list and compute the error rate associated to each
approximation. The `LatticeSurgery` layout links the cat-state layout to
the logical operation provided by the code.

The ISA query provides the specific options for the code, in this case
`ThreeAux` (an instance of the surface code). Magic states are not
provided by the code, and so the `RoundBasedFactory` is a generic model
that provides magic state distillation.

In [ ]:
from qsharp.qre import estimate, PSSPC, LatticeSurgery
from qsharp.qre.models import ThreeAux, RoundBasedFactory

trace_query = (
    app_query
    * PSSPC.q(num_ts_per_rotation=[16, 17, 18, 19])
    * LatticeSurgery.q()
)

isa_query = (
    ThreeAux.q(distance=[11, 13, 15, 17, 19])
    * RoundBasedFactory.q(
        code_query=ThreeAux.q(distance=[5, 7, 11, 13, 15, 17, 19])
    )
)

results = estimate(
    app, arch, isa_query, trace_query,
    max_error=0.01,
    name="Majorana e-5, 3-aux",
)

## Visualizing and Understanding the Results

### Result Summary Table

Canonically the number of physical qubits, runtime for a shot, and the
shot error rate are presented. QRE reports only on configurations of the
parameters that are Pareto-optimal in time and space (and satisfy the
specified error rate in the `estimate` command). We add additional columns
for information about these configurations.

Notice the tradeoff between space (number of qubits) and time (runtime) is
governed entirely by the selection of code distance. The higher code
distance means the algorithm runs more slowly and the compute register is
larger. But as a side effect we need far fewer magic state factories, and
hence the total qubit count goes down!

In [ ]:
from qsharp.qre.instruction_ids import LATTICE_SURGERY
from qsharp.qre.property_keys import (
    NUM_TS_PER_ROTATION, DISTANCE, PHYSICAL_COMPUTE_QUBITS,
)

results.add_column(
    "compute_distance",
    lambda entry: entry.source[LATTICE_SURGERY].instruction[DISTANCE],
)
results.add_column(
    "compute qubits",
    lambda entry: entry.properties[PHYSICAL_COMPUTE_QUBITS],
)
results.add_column(
    "num_ts_per_rotation",
    lambda entry: entry.properties[NUM_TS_PER_ROTATION],
)
results.add_factory_summary_column()
results.add_column(
    "cycle_time",
    lambda entry: entry.source[LATTICE_SURGERY].instruction.expect_time(1),
)

results.as_frame()

### Throttling the Algorithm

Seeing that we want the algorithm to run more slowly to conserve on qubit
count, we can simply increase the runtime of each lattice surgery step.
This can be done directly as an argument to the `LatticeSurgery` layout.
Again we provide a list of options; QRE will examine each of these and
report Pareto-optimal configurations.

In [ ]:
new_trace_query = (
    app_query
    * PSSPC.q(num_ts_per_rotation=[15, 16, 17, 18])
    * LatticeSurgery.q(
        slow_down_factor=[1.0 * j for j in range(2, 31)]
    )
)

new_results = estimate(
    app, arch, isa_query, new_trace_query,
    max_error=0.01,
    name="new Majorana e-5, 3-aux",
)
new_results.add_column(
    "compute_distance",
    lambda entry: entry.source[LATTICE_SURGERY].instruction[DISTANCE],
)
new_results.add_column(
    "compute qubits",
    lambda entry: entry.properties[PHYSICAL_COMPUTE_QUBITS],
)
new_results.add_column(
    "num_ts_per_rotation",
    lambda entry: entry.properties[NUM_TS_PER_ROTATION],
)
new_results.add_factory_summary_column()

new_results.as_frame()

## Tradeoff Curves

We can use matplotlib to create nice plots of our tradeoff curve.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

combined = pd.concat([results.as_frame(), new_results.as_frame()])
plt.figure(figsize=(10, 6))
for name, group in combined.groupby("name"):
    plt.scatter(group["qubits"], group["runtime"],
               marker="x", label=name)
plt.xlabel("Total Qubits")
plt.ylabel("Runtime (seconds)")
plt.legend()
plt.show()